# Remote MCP Servers: Connecting to O'Reilly

In this notebook, we connect an agent to a remote MCP server over HTTP. The server is O'Reilly's Search MCP, which searches the O'Reilly learning library and returns titles and links. We only describe where the server is and how to authenticate. The agent discovers the server's tools at runtime, and there is no tool code on our side.

## Setup

First, let's set up our environment:

In [ ]:
# Setup for running async code in Jupyter
import nest_asyncio
nest_asyncio.apply()

# Load optional environment variables from a .env file (if present).
# This course uses your Claude subscription, so no ANTHROPIC_API_KEY is needed --
# load_dotenv() is here only for optional keys (e.g. TAVILY_API_KEY), or an
# ANTHROPIC_API_KEY if you choose per-token API billing instead.
from dotenv import load_dotenv
load_dotenv()

print("✓ Notebook environment configured")

## Authentication

This course runs on your **Claude Pro/Max subscription** — no API key or per-token billing required.

**One-time setup:**

1. Install the Claude Code CLI. The Python SDK drives it under the hood (it is *not* bundled with `claude-agent-sdk`):
   ```bash
   # macOS / Linux (native installer)
   curl -fsSL https://claude.ai/install.sh | bash
   # or with npm:  npm install -g @anthropic-ai/claude-code
   ```
2. Log in with your Claude account:
   ```bash
   claude        # opens a browser to log in (use /login if not prompted)
   ```
   Inside `claude`, run `/status` to confirm you're authenticated via your subscription.

> ⚠️ **Precedence gotcha:** if `ANTHROPIC_API_KEY` is set in your environment, the SDK uses it (per-token API billing) and **ignores your subscription**. For subscription use, keep `ANTHROPIC_API_KEY` unset — don't put it in `.env`.

See the [Claude Code quickstart](https://code.claude.com/docs/en/quickstart) for Windows and other install options.

### Optional: use an API key instead

Prefer per-token API billing? You don't need to change any code — just make the key available in the environment and the SDK picks it up automatically:

```bash
# Option A - shell environment variable
export ANTHROPIC_API_KEY="sk-ant-..."
```

```bash
# Option B - add to a .env file in the project (the setup cell above calls load_dotenv())
# ANTHROPIC_API_KEY=sk-ant-...
```

```python
# (Optional) confirm which auth the SDK will use:
# import os
# print("API key billing" if os.environ.get("ANTHROPIC_API_KEY") else "Claude subscription")
```

### O'Reilly MCP token

O'Reilly's Search MCP needs a personal token. Create one on the MCP Tokens page of your O'Reilly account, then add it to the `.env` file in the repository root:

```bash
OREILLY_MCP_TOKEN=your_token_here
```

The token is separate from your Claude authentication. Keep it in `.env`, never in a notebook cell or a chat.

In [ ]:
import os

oreilly_token = os.environ.get("OREILLY_MCP_TOKEN")
if oreilly_token:
    print("OREILLY_MCP_TOKEN is set")
else:
    print("OREILLY_MCP_TOKEN is not set. Add it to .env, then rerun the setup cell and this one.")

## Helper Function

Let's create a helper to print messages cleanly:

In [ ]:
import json

def print_message(message):
    """Pretty print agent messages."""
    msg_type = type(message).__name__
    
    if msg_type == "SystemMessage":
        pass  # Skip system messages
    
    elif msg_type == "AssistantMessage":
        if hasattr(message, 'content'):
            for block in message.content:
                block_type = type(block).__name__
                if block_type == "TextBlock":
                    print(f"🤖 Assistant: {block.text}")
                elif block_type == "ToolUseBlock":
                    print(f"🔧 Tool: {block.name}")
                    if hasattr(block, 'input') and 'description' in block.input:
                        print(f"   → {block.input['description']}")
    
    elif msg_type == "UserMessage":
        if hasattr(message, 'content'):
            for block in message.content:
                block_type = type(block).__name__
                if block_type == "ToolResultBlock":
                    if block.is_error:
                        print(f"❌ Tool Error: {block.content}")
                    else:
                        content = str(block.content)
                        if len(content) > 300:
                            content = content[:300] + "..."
                        print(f"📤 Tool Result: {content}")
    
    elif msg_type == "ResultMessage":
        if hasattr(message, 'total_cost_usd') and hasattr(message, 'duration_ms'):
            print(f"\n💰 Cost: ${message.total_cost_usd:.4f} | ⏱️ Time: {message.duration_ms/1000:.1f}s")

## Local and Remote MCP Servers

The SDK can reach MCP servers in three ways:

- **In-process** (`create_sdk_mcp_server`): tools you write in Python that run inside your application
- **Local** (`stdio`): a program the SDK starts on your machine, like the Tavily server in the Module 3 exercise
- **Remote** (`http` or `sse`): a server someone else runs, reached over the network

A remote server needs a URL, plus headers when it requires authentication. The token goes in the `Authorization` header of each request to the server, so it never enters the model's context.

In [ ]:
oreilly_server = {
    "type": "http",
    "url": "https://api.oreilly.com/api/content-discovery/v1/mcp/",
    "headers": {"Authorization": f"Bearer {oreilly_token}"},
}

## Example 1: Discovering the Server's Tools

We don't list the tools ourselves. When a session starts, the SDK connects to each configured server and asks it for its tools. `get_mcp_status()` shows the connection status, the server's name and version, and the tools it offers. Servers can annotate their tools, for example to mark one as read-only.

`strict_mcp_config=True` makes the SDK use only the servers we pass in `mcp_servers`. Without it, the agent would also get any MCP servers or connectors you set up for Claude Code or claude.ai.

Remote servers connect in the background, so the code checks the status until the server is no longer pending.

In [ ]:
import asyncio
from claude_agent_sdk import ClaudeSDKClient, ClaudeAgentOptions

async def show_oreilly_tools():
    options = ClaudeAgentOptions(
        mcp_servers={"oreilly": oreilly_server},
        strict_mcp_config=True,
    )

    async with ClaudeSDKClient(options) as client:
        # Wait up to 10 seconds for the remote server to connect
        for _ in range(20):
            status = await client.get_mcp_status()
            if all(server["status"] != "pending" for server in status["mcpServers"]):
                break
            await asyncio.sleep(0.5)

    for server in status["mcpServers"]:
        info = server.get("serverInfo", {})
        print(f"{server['name']}: {server['status']} ({info.get('name', '')} {info.get('version', '')})")
        if server.get("error"):
            print(f"  Error: {server['error']}")
        for tool in server.get("tools", []):
            print(f"  - mcp__{server['name']}__{tool['name']}  annotations: {tool.get('annotations', {})}")

await show_oreilly_tools()

### Tool Names

Each tool from an MCP server is exposed to Claude as `mcp__<server>__<tool>`, where `<server>` is the key we used in `mcp_servers`. A wildcard such as `mcp__oreilly__*` allows every tool from that server.

## Example 2: Searching O'Reilly with the Agent

Now we give the agent the server and allow its tools. The agent decides when to search and how to use the results.

In [ ]:
from claude_agent_sdk import query

async def find_oreilly_content():
    options = ClaudeAgentOptions(
        mcp_servers={"oreilly": oreilly_server},
        allowed_tools=["mcp__oreilly__*"],
        strict_mcp_config=True,
        max_turns=10,
    )

    prompt = (
        "Find O'Reilly content to help me build AI agents with the Claude Agent SDK. "
        "Recommend four items, mixing live courses, books, and videos, each with its link."
    )
    async for message in query(prompt=prompt, options=options):
        print_message(message)

await find_oreilly_content()

### What Just Happened?

- The SDK connected to O'Reilly's server over HTTP and sent the token in the `Authorization` header
- The agent loaded the O'Reilly search tool on demand with ToolSearch (MCP tool search, slide 115), then called it with queries it wrote from our request
- The tool results came back as text, and the agent turned them into recommendations with links
- `allowed_tools` approved the O'Reilly tools without a permission prompt